In [2]:
import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer, util
import yake


INPUT_FILE = "dataset_sample_10k.json"
OUTPUT_FILE = "coverage_results_rich.json"

# the top 5 concepts
TOP_K = 5

# analyze over a set of thresholds
THRESHOLDS = [0.6, 0.7, 0.8]

# the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# setting up YAKE
kw_extractor = yake.KeywordExtractor(
    lan="en",
    n=2,
    dedupLim=0.9,
    top=TOP_K
)

# keywords normalization and extraction
def normalize(text):
    return text.lower().strip()

def normalize_keywords(keywords):
    if isinstance(keywords, list):
        return [normalize(k) for k in keywords if isinstance(k, str)]
    elif isinstance(keywords, str):
        return [normalize(keywords)]
    return []

def extract_keywords(text):
    kws = kw_extractor.extract_keywords(text)
    return [normalize(k[0]) for k in kws]

# -------- MAIN --------
if __name__ == "__main__":

    start_time = time.time()

    print("🔄 Loading dataset...\n")
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Loaded {len(data)} datasets\n")

    results = []

    for i, d in enumerate(data, 1):

        title = d.get("title", "")
        description = d.get("description", "")

        keywords = normalize_keywords(d.get("keywords", []))

        # Extract concepts separately
        extracted_title = extract_keywords(title)
        extracted_description = extract_keywords(description)

        # Default values
        title_coverage_dict = {
            f"title_coverage_{t}": 0.0 for t in THRESHOLDS
        }

        description_coverage_dict = {
            f"description_coverage_{t}": 0.0 for t in THRESHOLDS
        }

        avg_title_max_similarity = 0.0
        avg_description_max_similarity = 0.0

        # TITLE COVERAGE
        if len(extracted_title) > 0 and len(keywords) > 0:

            extracted_title_embs = model.encode(
                extracted_title,
                convert_to_tensor=True
            )

            keyword_embs = model.encode(
                keywords,
                convert_to_tensor=True
            )

            sims_title = util.cos_sim(
                extracted_title_embs,
                keyword_embs
            )

            # Coverage for thresholds
            for t in THRESHOLDS:

                matches = (sims_title >= t).any(dim=1)

                coverage_t = float(
                    matches.sum().item() / len(extracted_title)
                )

                title_coverage_dict[
                    f"title_coverage_{t}"
                ] = coverage_t

            # Average max similarity
            max_sims_title = sims_title.max(dim=1).values

            avg_title_max_similarity = float(
                max_sims_title.mean().item()
            )

        # DESCRIPTION COVERAGE
        if len(extracted_description) > 0 and len(keywords) > 0:

            extracted_description_embs = model.encode(
                extracted_description,
                convert_to_tensor=True
            )

            keyword_embs = model.encode(
                keywords,
                convert_to_tensor=True
            )

            sims_description = util.cos_sim(
                extracted_description_embs,
                keyword_embs
            )

            # Coverage for thresholds
            for t in THRESHOLDS:

                matches = (sims_description >= t).any(dim=1)

                coverage_t = float(
                    matches.sum().item() / len(extracted_description)
                )

                description_coverage_dict[
                    f"description_coverage_{t}"
                ] = coverage_t

            # Average max similarity
            max_sims_description = sims_description.max(dim=1).values

            avg_description_max_similarity = float(
                max_sims_description.mean().item()
            )

        # SAVE RESULTS
        results.append({
            "dataset_id": d.get("dataset_id"),

            "num_keywords": len(keywords),

            "num_title_concepts": len(extracted_title),
            "num_description_concepts": len(extracted_description),

            "avg_title_max_similarity": avg_title_max_similarity,
            "avg_description_max_similarity": avg_description_max_similarity,

            **title_coverage_dict,
            **description_coverage_dict
        })

        if (i + 1) % 500 == 0:
            print(f"Processed {i+1}/{len(data)}")

    # -------- ROBUSTNESS SUMMARY --------
    robustness_summary = {}

    for t in THRESHOLDS:
        robustness_summary[f"title_coverage_mean_{t}"] = float(
            np.mean([r[f"title_coverage_{t}"] for r in results])
        )

        robustness_summary[f"description_coverage_mean_{t}"] = float(
            np.mean([r[f"description_coverage_{t}"] for r in results])
        )

    # -------- SAVE --------
    output_payload = {
        "dataset_results": results,
        "robustness_summary": robustness_summary
    }

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output_payload, f, indent=2)

    end_time = time.time()

    print(f"\n💾 Results saved to {OUTPUT_FILE}")
    print(f"\n⏱ Total time: {end_time - start_time:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading dataset...

Loaded 10000 datasets

Processed 500/10000
Processed 1000/10000
Processed 1500/10000
Processed 2000/10000
Processed 2500/10000
Processed 3000/10000
Processed 3500/10000
Processed 4000/10000
Processed 4500/10000
Processed 5000/10000
Processed 5500/10000
Processed 6000/10000
Processed 6500/10000
Processed 7000/10000
Processed 7500/10000
Processed 8000/10000
Processed 8500/10000
Processed 9000/10000
Processed 9500/10000
Processed 10000/10000

💾 Results saved to coverage_results_rich.json

⏱ Total time: 4098.27s
